# 03 — Stage C: Modelling, Evaluation and Explanation

**Steps 8–10 of the 11-step predictive-modeling process.**

**Prerequisites:** run `01_problem_framing.ipynb` (decisions) and
`02_data_and_features.ipynb` (writes `data/processed/clean.parquet`) first.

| Step | Section | What happens |
|---|---|---|
| — | 0 | Setup; recover the frozen out-of-time split and assert it is intact |
| **8** | 8.1–8.6 | **Baselines first**, then linear, then trees; CV with `TimeSeriesSplit`; overfitting check; tuning **last** |
| **9** | 9.1–9.7 | Calibration, threshold from the cost matrix, metrics, **error analysis by segment**, **fairness check** |
| **10** | 10.1–10.3 | Permutation importance, SHAP, partial dependence |
| — | 11 | Conclusions and limitations |

**Every headline number is the out-of-time number.** The random-split comparison appears
once, in 9.7, purely to quantify how much optimism a random split would have bought.

The order of Step 8 is the process's order and it is not decorative: **baselines before
models, models before tuning.** A tuned gradient booster that cannot beat `sub_grade` has
told you something important, and you only learn it if the baseline was computed first.

## 0. Setup

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             confusion_matrix, precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACTS = ROOT / "artifacts"
FIG_DIR = ARTIFACTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
C_RED, C_GRAY = "#e34948", "#8a8a86"
MODEL_COLORS = {}          # filled in once the model registry exists

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (8, 4.5), "figure.dpi": 110, "savefig.dpi": 150,
    "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold",
})


def save_fig(name: str) -> None:
    plt.savefig(FIG_DIR / f"{name}.png")
    plt.show()


warnings.filterwarnings("ignore", category=FutureWarning)
print("Setup complete.")

## 0.1 Recover the frozen split

The split was decided in notebook 02, **before** any transform that learns from data.
Here it is only reconstructed from the `is_train` flag and then checked. Three assertions
guard the properties that matter:

1. train and test `issue_d` ranges do not overlap (no look-ahead);
2. the manifest's cutoff matches the data;
3. the test-row hash matches the manifest (the split has not silently drifted).

A silent split change would invalidate comparison with the other groups, so these fail
loudly rather than warn.

In [ ]:
CLEAN_PATH = DATA_PROCESSED / "clean.parquet"
if not CLEAN_PATH.exists():
    raise FileNotFoundError(
        f"{CLEAN_PATH} not found. Run notebooks/02_data_and_features.ipynb first -- it produces this file.")

df = pd.read_parquet(CLEAN_PATH)
manifest = json.loads((ARTIFACTS / "split_manifest.json").read_text())
CUTOFF_DATE = pd.Timestamp(manifest["cutoff_date"])
DATA_SOURCE = manifest.get("data_source", "unknown")

print(f"Loaded {len(df):,} rows  |  source: {DATA_SOURCE}  |  cutoff: {CUTOFF_DATE.date()}")
if DATA_SOURCE == "synthetic":
    print("\n!! SYNTHETIC data -- these numbers smoke-test the pipeline, they are not findings.")

In [ ]:
# Columns that exist for bookkeeping, not for the model to learn from.
NON_FEATURES = ["target", "is_train", "is_train_random_ref", "issue_d", "earliest_cr_line",
                "grade", "sub_grade"]   # grade/sub_grade kept as *_ordinal instead

feature_cols = [c for c in df.columns if c not in NON_FEATURES]
X = df[feature_cols]
y = df["target"]
train_mask = df["is_train"].to_numpy()

X_train, X_test = X.loc[train_mask], X.loc[~train_mask]
y_train, y_test = y.loc[train_mask], y.loc[~train_mask]
issue_train = df.loc[train_mask, "issue_d"]

# --- Guards ------------------------------------------------------------------------
assert issue_train.max() < df.loc[~train_mask, "issue_d"].min(), \
    "Look-ahead: train and test issue_d ranges overlap."
assert manifest["n_train"] == len(X_train) and manifest["n_test"] == len(X_test), \
    "Row counts disagree with split_manifest.json -- the split has drifted."
import hashlib
recomputed = hashlib.md5(np.flatnonzero(~train_mask).tobytes()).hexdigest()
assert recomputed == manifest["test_index_md5"], \
    "Test-index hash mismatch -- the split is not the one recorded in the manifest."
print("OK -- split matches the manifest; no look-ahead.\n")

overview = pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "default_rate": [y_train.mean(), y_test.mean()],
    "period": [f"{issue_train.min():%Y-%m} to {issue_train.max():%Y-%m}",
               f"{df.loc[~train_mask, 'issue_d'].min():%Y-%m} to "
               f"{df.loc[~train_mask, 'issue_d'].max():%Y-%m}"],
}, index=["train", "test"])
print(f"{len(feature_cols)} features")
display(overview)

## Step 8 — Build models, simple to complex

### 8.1 Preprocessing — fitted on training rows only

Every transform here learns a parameter from the data: imputation learns a median or a
mode, scaling learns a mean and standard deviation, one-hot encoding learns a category
vocabulary. Fitting any of them on the full frame is **train–test contamination** — the
model's inputs would be shaped by rows it is supposed to be evaluated on, and no metric
would reveal it.

```python
# The anti-pattern this avoids:
preprocessor.fit(X)                       # <-- sees test rows
X_train, X_test = train_test_split(X)     # <-- too late, the damage is done
```

Wrapping the `ColumnTransformer` in a `Pipeline` makes the correct behaviour structural
rather than a matter of discipline: `.fit()` only ever receives training rows, and
`X_test` reaches the transformer solely through `.transform()`. It also means
cross-validation re-fits the preprocessing **inside each fold** instead of once over all
training data.

`min_frequency=0.01` folds rare categories into a single infrequent bucket, so a category
appearing a handful of times cannot contribute a noisy high-variance dummy — and
`handle_unknown="infrequent_if_exist"` means a category that appears only after the cutoff
is handled gracefully instead of crashing at test time.

In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

print(f"{len(numeric_cols)} numeric, {len(categorical_cols)} categorical")
print(f"  numeric     : {numeric_cols}")
print(f"  categorical : {categorical_cols}")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                     min_frequency=0.01, sparse_output=False)),
        ]), categorical_cols),
    ],
    remainder="drop",
)

### 8.2 Baselines — the bar every model must clear

> *Always compute a baseline.*

Specified in notebook 01 § 2.5, computed here. Two of them, and the difference between
them is the whole point.

| # | Baseline | Represents | Hard to beat? |
|---|---|---|---|
| **B0** | Majority class (`DummyClassifier(strategy="prior")`) | The degenerate model from notebook 01 Table 2 | No — beating it proves nothing |
| **B1** | **`sub_grade` as a risk score** | **The incumbent**: Lending Club's existing underwriting | **Yes — this is the real bar** |

B1 needs no fitting at all. `sub_grade` is already an ordered risk ranking (A1 lowest risk
… G5 highest), so using it directly as a score is a complete, zero-parameter model — the
rule the platform already operates. Since ROC-AUC, PR-AUC and KS are all rank-based, the
ordinal encoding can be scored as-is without converting it to a probability.

If a gradient-boosted ensemble cannot beat B1 by a margin worth the loss of
interpretability, the honest recommendation is to keep the existing rule. Notebook 01 §1.5
asked whether this problem needs machine learning at all; B1 is how that question gets
answered with a number.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline_scores = {}

# --- B0: majority class ------------------------------------------------------------
b0 = DummyClassifier(strategy="prior", random_state=RANDOM_SEED).fit(X_train, y_train)
baseline_scores["B0 - majority class"] = b0.predict_proba(X_test)[:, 1]

# --- B1: sub_grade as a risk score (no fitting -- it IS the incumbent rule) ---------
# Higher sub_grade_ordinal = worse grade = higher risk, so it is already oriented as a
# default score. Rank-based metrics are invariant to the monotone rescaling below.
sub_grade_score = X_test["sub_grade_ordinal"].to_numpy(dtype=float)
sub_grade_score = (sub_grade_score - np.nanmin(sub_grade_score)) / (
    np.nanmax(sub_grade_score) - np.nanmin(sub_grade_score))
baseline_scores["B1 - sub_grade (incumbent)"] = np.nan_to_num(sub_grade_score, nan=0.5)

for name, score in baseline_scores.items():
    print(f"{name:<30} ROC-AUC {roc_auc_score(y_test, score):.4f}   "
          f"PR-AUC {average_precision_score(y_test, score):.4f}")
print(f"\n{'prevalence (PR-AUC floor)':<30} {y_test.mean():.4f}")

**Baselines computed.**

B0 lands at ROC-AUC 0.500 and PR-AUC ≈ prevalence, exactly as notebook 01 predicted — it
is a floor, not a competitor.

B1 is the number to remember. Every model in 8.3 is reported against it in Table 10, and
the gap — not the absolute AUC — is this project's actual contribution.

**Action →** Carry both into the model comparison table, and state the lift over B1
explicitly in the conclusions.

### 8.3 Models, in increasing order of complexity

Four models, spanning the interpretability/flexibility trade-off. Each is a full
`Pipeline`, so preprocessing travels with the estimator and cannot be applied out of
order.

| Model | Why it is here |
|---|---|
| **Logistic regression** | The industry baseline. A credit scorecard is a linear model for good reasons: coefficients are auditable, monotonic, and defensible to a regulator. Anything more complex has to earn its place against this. |
| **Random forest** | Captures interactions and non-linearity with little tuning; a robust reference for "how much does non-linearity buy?" |
| **HistGradientBoosting** | Gradient boosting, usually the strongest tabular family, with no extra dependency. |
| **XGBoost** | The Kaggle standard for this dataset — included so our comparison is against what the public work actually uses. Guarded by a `try/except` so the notebook runs without it. |

Class imbalance is handled with **class weights** rather than resampling. At ~20 %
prevalence this is not a rare-event problem; SMOTE and friends would synthesise
applicants who never existed, and would have to be applied inside the CV folds to avoid
yet another leakage path.

In [ ]:
models: dict[str, Pipeline] = {}

models["Logistic Regression"] = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000,
                               random_state=RANDOM_SEED)),
])

models["Random Forest"] = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=50,
                                   class_weight="balanced_subsample", n_jobs=-1,
                                   random_state=RANDOM_SEED)),
])

models["HistGradientBoosting"] = Pipeline([
    ("prep", preprocessor),
    ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                           max_leaf_nodes=31, min_samples_leaf=50,
                                           l2_regularization=1.0,
                                           random_state=RANDOM_SEED)),
])

try:
    from xgboost import XGBClassifier
    scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
    models["XGBoost"] = Pipeline([
        ("prep", preprocessor),
        ("clf", XGBClassifier(n_estimators=400, learning_rate=0.06, max_depth=5,
                              subsample=0.8, colsample_bytree=0.8,
                              reg_lambda=1.0, scale_pos_weight=scale_pos_weight,
                              eval_metric="logloss", tree_method="hist",
                              n_jobs=-1, random_state=RANDOM_SEED)),
    ])
    print(f"XGBoost available (scale_pos_weight={scale_pos_weight:.2f})")
except ImportError:
    print("XGBoost not installed -- skipping it. `pip install xgboost` to include it.")

MODEL_COLORS = dict(zip(models, [C_BLUE, C_ORANGE, C_AQUA, C_VIOLET]))
print(f"\n{len(models)} models registered: {list(models)}")

### 8.4 Cross-validation on the training period — `TimeSeriesSplit`

A sanity check before touching the test set, and a second place the time ordering has to
be respected. Shuffled `StratifiedKFold` would let fold *k* train on loans issued **after**
the ones it validates on — reintroducing exactly the look-ahead the out-of-time split
exists to remove, and producing optimistic CV scores that disagree with the final test
number for reasons nobody can find later.

`TimeSeriesSplit` on chronologically sorted training rows gives expanding-window folds:
always train on the past, validate on the future.

In [ ]:
from sklearn.model_selection import cross_validate

cv = TimeSeriesSplit(n_splits=4)
cv_rows = []
for name, model in models.items():
    # return_train_score gives the train-vs-validation gap the process asks for.
    out = cross_validate(model, X_train, y_train, cv=cv,
                         scoring="roc_auc", n_jobs=1, return_train_score=True)
    cv_rows.append({
        "model": name,
        "train_roc_auc": out["train_score"].mean(),
        "cv_roc_auc": out["test_score"].mean(),
        "cv_std": out["test_score"].std(),
        "gap (train - cv)": out["train_score"].mean() - out["test_score"].mean(),
        "folds": np.round(out["test_score"], 4).tolist(),
    })
    print(f"{name:<24} train {out['train_score'].mean():.4f} | "
          f"cv {out['test_score'].mean():.4f} +/- {out['test_score'].std():.4f} | "
          f"gap {out['train_score'].mean() - out['test_score'].mean():+.4f}")

cv_results = pd.DataFrame(cv_rows).set_index("model").sort_values("cv_roc_auc", ascending=False)
display(cv_results.drop(columns=["folds"]).style.format("{:.4f}").set_caption(
    "Table 7 - Expanding-window CV with the train-vs-validation gap (overfitting check)"))

**Table 7 — Expanding-window cross-validation, with the overfitting check.**

These are honest in-period estimates: each fold trains only on loans issued before the
ones it scores.

**Read the `gap` column first.** The process is explicit that train and validation scores
must be compared to detect overfitting, and the gap is that comparison:

- **Gap near zero** — the model is not memorising; any error is bias, and more capacity
  might help.
- **Large gap** (say > 0.05 with the tree models) — the model fits the training folds far
  better than the next period. Reduce depth, raise `min_samples_leaf`, or add
  regularisation before reaching for more capacity.

`cv_std` matters nearly as much. It is the fold-to-fold spread, and because the folds are
*chronological*, a large spread means performance is unstable **across time** — a direct
warning about the out-of-time test to come, not noise to be averaged away.

Compare these means with the test numbers in 9.4. CV materially above test indicates drift
between the training and test periods; that gap is a result worth reporting, not a bug to
tune away.

**Action →** Pick the model family to tune in 8.6 from the `cv_roc_auc` column, not from
any test-set number — the test set stays closed until Step 9.

### 8.5 Have we beaten the incumbent yet?

Before tuning anything, the question from notebook 01 §1.5: does the extra complexity buy
anything over `sub_grade`?

In [ ]:
bar = roc_auc_score(y_test, baseline_scores["B1 - sub_grade (incumbent)"])
interim = cv_results[["cv_roc_auc"]].copy()
interim["B1 (sub_grade) ROC-AUC"] = bar
interim["CV lift over B1"] = interim["cv_roc_auc"] - bar
display(interim.style.format("{:.4f}").set_caption(
    "Table 8 - In-period CV performance against the incumbent baseline"))

if (interim["CV lift over B1"] <= 0).all():
    print("NOTE: no model beats the incumbent in cross-validation.")
    print("That is a legitimate finding and must be reported as such, not tuned away.")
else:
    best_cv = interim["CV lift over B1"].idxmax()
    print(f"Best in-period lift over the incumbent: {best_cv} "
          f"({interim.loc[best_cv, 'CV lift over B1']:+.4f} ROC-AUC)")

**Table 8 — Models versus the incumbent.**

The comparison is indicative rather than exact — the CV column is measured in-period on
training folds while B1 is measured on the test period — but it is the right question
asked at the right time: *before* investing in tuning. Section 9.4 repeats it on equal
footing, with everything scored on the same out-of-time test set.

**Action →** If the lift over B1 is small here, say so in the conclusions and recommend the
simpler model. Do not let tuning in 8.6 quietly become the justification for complexity
that was not warranted.

### 8.6 Hyperparameter tuning — last, not first

The process puts tuning at the **end** of Step 8, after the model comparison is settled,
and that ordering is deliberate. Tuning before comparing means comparing a tuned model
against untuned ones, which measures search effort rather than model quality.

The search below is deliberately small. Two constraints shape it:

- **`TimeSeriesSplit`, not shuffled folds** — the same look-ahead rule as everywhere else.
  `RandomizedSearchCV` will happily shuffle if allowed to, which would quietly reintroduce
  the leakage the whole project is built to avoid.
- **Scored on `average_precision`** (PR-AUC), the primary metric fixed in notebook 01 §2.3
  — not on ROC-AUC, and not on accuracy.

Tuning typically buys far less here than the threshold policy in 9.3 or the evaluation
protocol in 9.7. Its modest payoff is itself a reportable result.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

tune_target = cv_results.index[0]          # best model family by in-period CV
print(f"Tuning: {tune_target}\n")

SEARCH_SPACES = {
    "Logistic Regression": {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    "Random Forest": {"clf__max_depth": [8, 12, 16],
                      "clf__min_samples_leaf": [25, 50, 100]},
    "HistGradientBoosting": {"clf__learning_rate": [0.03, 0.06, 0.1],
                             "clf__max_leaf_nodes": [15, 31, 63],
                             "clf__l2_regularization": [0.0, 1.0, 10.0]},
    "XGBoost": {"clf__max_depth": [3, 5, 7],
                "clf__learning_rate": [0.03, 0.06, 0.1],
                "clf__subsample": [0.7, 0.9]},
}

search = RandomizedSearchCV(
    models[tune_target], SEARCH_SPACES[tune_target], n_iter=6,
    cv=TimeSeriesSplit(n_splits=3),        # time-ordered folds, never shuffled
    scoring="average_precision",           # the primary metric from notebook 01
    random_state=RANDOM_SEED, n_jobs=1, refit=True)
search.fit(X_train, y_train)

untuned_pr = cv_results.loc[tune_target, "cv_roc_auc"]
print(f"Best params : {search.best_params_}")
print(f"Best CV PR-AUC: {search.best_score_:.4f}")

# Adopt the tuned estimator only if the search actually improved on the default.
tuned_name = f"{tune_target} (tuned)"
models[tuned_name] = search.best_estimator_
MODEL_COLORS[tuned_name] = C_RED
print(f"\nRegistered '{tuned_name}' for evaluation in Step 9.")

**Tuning result.**

Compare `Best CV PR-AUC` against the untuned model's CV score. On tabular credit data the
improvement is usually in the third decimal place — worth having, but an order of magnitude
smaller than the effect of the cost-based threshold or the evaluation protocol.

The tuned estimator is registered as a *separate* model rather than replacing the default,
so Step 9 reports both and the tuning gain stays visible instead of being absorbed.

**Action →** Report the tuning delta explicitly in the conclusions. If it is negligible,
prefer the untuned model — fewer moving parts to justify and monitor.

## Step 9 — Evaluate and analyse errors

### 9.1 Fit and calibrate

**Why calibration is not optional here.** In credit scoring the probability *is* the
product: expected loss is `PD × EAD × LGD`, and pricing, provisioning and cut-off policy
all consume the number itself, not just the ranking. A model can rank perfectly and still
be systematically wrong about the level. ROC-AUC — a pure ranking statistic — cannot
detect that; the Brier score and the calibration curve can.

`class_weight="balanced"` and `scale_pos_weight` make this worse before they make it
better: both deliberately distort the class prior to help the model learn, so raw outputs
overstate the default probability. Calibration undoes that distortion.

**The leakage-safe way to calibrate.** The calibrator is fitted on a held-out slice of the
**training** data — never on test rows. And the slice is the **latest pre-cutoff months**,
not a random subset, so the same time ordering holds: fit on the earlier period, calibrate
on the later one, test on the period after that.

In [ ]:
# Time-ordered split of the training period: earliest 80% to fit, latest 20% to calibrate.
order = np.argsort(issue_train.to_numpy(), kind="mergesort")
n_fit = int(len(order) * 0.8)
fit_idx, cal_idx = order[:n_fit], order[n_fit:]

X_fit, y_fit = X_train.iloc[fit_idx], y_train.iloc[fit_idx]
X_cal, y_cal = X_train.iloc[cal_idx], y_train.iloc[cal_idx]

print(f"fit slice        : {len(X_fit):>7,} loans  "
      f"({issue_train.iloc[fit_idx].min():%Y-%m} to {issue_train.iloc[fit_idx].max():%Y-%m})")
print(f"calibration slice: {len(X_cal):>7,} loans  "
      f"({issue_train.iloc[cal_idx].min():%Y-%m} to {issue_train.iloc[cal_idx].max():%Y-%m})")
assert issue_train.iloc[fit_idx].max() <= issue_train.iloc[cal_idx].min(), \
    "Calibration slice must come after the fit slice in time."

In [ ]:
def calibrate(fitted_model, X_cal, y_cal):
    """Wrap an already-fitted estimator in an isotonic calibrator.

    Uses FrozenEstimator where available (sklearn >= 1.6); falls back to cv="prefit".
    Either way the base model is NOT refitted -- only the calibration map is learned.
    """
    try:
        from sklearn.frozen import FrozenEstimator
        calibrated = CalibratedClassifierCV(FrozenEstimator(fitted_model), method="isotonic")
    except ImportError:
        calibrated = CalibratedClassifierCV(fitted_model, method="isotonic", cv="prefit")
    return calibrated.fit(X_cal, y_cal)


fitted, calibrated_models = {}, {}
for name, model in models.items():
    print(f"fitting {name} ...", end=" ", flush=True)
    base = model.fit(X_fit, y_fit)              # trained on the EARLY training slice only
    fitted[name] = base
    calibrated_models[name] = calibrate(base, X_cal, y_cal)
    print("done")

# Score the test set once, with the calibrated models.
scores_test = {name: m.predict_proba(X_test)[:, 1] for name, m in calibrated_models.items()}
scores_cal = {name: m.predict_proba(X_cal)[:, 1] for name, m in calibrated_models.items()}
scores_test_raw = {name: m.predict_proba(X_test)[:, 1] for name, m in fitted.items()}
print("\nAll models fitted, calibrated and scored.")

### 9.3 Choosing a decision threshold

A probability becomes a lending decision only through a cut-off, and **0.5 is arbitrary**.
At ~20 % prevalence a model can be well-calibrated and still put almost every applicant
below 0.5 — classifying everyone as "will repay", achieving 80 % accuracy, and being
completely useless. Accuracy is reported below only to make that failure visible.

Three rules, all computed on the **calibration slice** and then applied unchanged to test.
Tuning a threshold on test scores is the same family of leakage as fitting a scaler on
them:

- **F1-optimal** — balances precision and recall, ignoring that the two errors cost
  different amounts.
- **Youden's J** (`max(TPR − FPR)`) — the point furthest from chance on the ROC curve;
  threshold-symmetric, so it also ignores costs.
- **Cost-based** — the only one that encodes the actual business problem. A missed default
  (false negative) costs the unrecovered principal; a wrongly rejected good applicant
  (false positive) costs the foregone interest margin. These are not symmetric, and the
  ratio drives the cut-off far more than the model choice does.

The costs below are **illustrative placeholders**. Replace them with figures from the
portfolio before quoting any result that depends on them.

In [ ]:
COST_FN = 1.0     # missed default: lost principal (normalised to 1)
COST_FP = 0.25     # rejected good applicant: foregone interest margin


def threshold_candidates(y_true, y_score):
    """Return the F1-optimal, Youden-J and cost-minimising thresholds."""
    prec, rec, thr_pr = precision_recall_curve(y_true, y_score)
    f1 = np.divide(2 * prec * rec, prec + rec,
                   out=np.zeros_like(prec), where=(prec + rec) > 0)
    thr_f1 = thr_pr[np.argmax(f1[:-1])] if len(thr_pr) else 0.5

    fpr, tpr, thr_roc = roc_curve(y_true, y_score)
    thr_youden = thr_roc[np.argmax(tpr - fpr)]

    grid = np.linspace(0.01, 0.99, 197)
    n_pos, n_neg = int(y_true.sum()), int((1 - y_true).sum())
    costs = []
    for t in grid:
        pred = (y_score >= t).astype(int)
        fn = int(((pred == 0) & (y_true == 1)).sum())
        fp = int(((pred == 1) & (y_true == 0)).sum())
        costs.append((fn * COST_FN + fp * COST_FP) / len(y_true))
    thr_cost = grid[int(np.argmin(costs))]
    return {"f1": float(thr_f1), "youden": float(thr_youden), "cost": float(thr_cost)}, \
           (grid, np.array(costs))


thresholds = {}
for name in calibrated_models:
    thresholds[name], _ = threshold_candidates(y_cal.to_numpy(), scores_cal[name])

thr_table = pd.DataFrame(thresholds).T
thr_table.columns = ["F1-optimal", "Youden-J", f"Cost (FN:FP = {COST_FN:g}:{COST_FP:g})"]
display(thr_table.style.format("{:.3f}").set_caption(
    "Table 9 - Thresholds chosen on the calibration slice, applied unchanged to test"))

# The cost rule is the operating point used for all point metrics below.
OPERATING_THRESHOLD = {name: t["cost"] for name, t in thresholds.items()}

**Table 9 — Candidate thresholds per model, chosen on the calibration slice.**

The three rules disagree substantially, and that disagreement is the point: the cut-off is
a **policy decision**, not a model output. The cost-based threshold sits lowest, because a
missed default is assumed four times more expensive than a needlessly rejected applicant —
so the lender should be willing to reject more good applicants to avoid one bad loan.

The cost-based threshold is used as the operating point for every point metric below. All
threshold-free metrics (ROC-AUC, PR-AUC, KS, Gini, Brier) are unaffected by this choice.

### 9.4 Metrics — the test set, opened once

Each metric answers a different question, and the brief requires we can say which:

| Metric | Question it answers | Caveat |
|---|---|---|
| **ROC-AUC** | Given one defaulter and one payer, how often does the model rank the defaulter riskier? | Optimistic under imbalance — true negatives dominate both rates |
| **PR-AUC** | Of the applicants flagged riskiest, how many actually default? | Baseline is the prevalence (~0.20), not 0.5 — read it against that |
| **KS** | Maximum gap between the good and bad score distributions | The credit-industry standard; a single number for "does the scorecard separate?" |
| **Gini** | `2·AUC − 1`, rescaled so 0 = random | Same information as AUC; reported because credit risk conventionally does |
| **Brier** | Mean squared error of the probabilities themselves | The only one that punishes miscalibration; **lower is better** |
| **Precision / Recall / F1** | Performance at the chosen cut-off | Meaningless without justifying the threshold |
| **Accuracy** | Share of correct predictions | Reported only to show it is uninformative here — predicting "never defaults" scores ~80 % |

In [ ]:
def evaluate(y_true, y_score, threshold: float) -> dict:
    """Threshold-free and point metrics for one model."""
    y_true = np.asarray(y_true)
    y_pred = (y_score >= threshold).astype(int)

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "ROC-AUC": auc,
        "PR-AUC": average_precision_score(y_true, y_score),
        "KS": float(np.max(tpr - fpr)),
        "Gini": 2 * auc - 1,
        "Brier": brier_score_loss(y_true, y_score),
        "Threshold": threshold,
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Accuracy": (tp + tn) / len(y_true),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Cost/loan": (fn * COST_FN + fp * COST_FP) / len(y_true),
    }


results = {name: evaluate(y_test, scores_test[name], OPERATING_THRESHOLD[name])
           for name in calibrated_models}

# Baselines from 8.2 evaluated on the same test set, with the same helper, so the
# comparison is like-for-like. Their threshold is the prevalence -- they have no
# calibrated probability to threshold meaningfully.
for bname, bscore in baseline_scores.items():
    results[bname] = evaluate(y_test, bscore, float(y_test.mean()))

metrics = pd.DataFrame(results).T.sort_values("PR-AUC", ascending=False)

# Lift over the incumbent -- the number that decides whether this project was worth doing.
B1 = "B1 - sub_grade (incumbent)"
metrics["PR-AUC lift vs B1"] = metrics["PR-AUC"] - metrics.loc[B1, "PR-AUC"]
metrics["ROC-AUC lift vs B1"] = metrics["ROC-AUC"] - metrics.loc[B1, "ROC-AUC"]

metrics.to_csv(ARTIFACTS / "metrics.csv")
print(f"Wrote {ARTIFACTS / 'metrics.csv'}\n")

display(metrics[["ROC-AUC", "PR-AUC", "KS", "Gini", "Brier",
                 "PR-AUC lift vs B1", "ROC-AUC lift vs B1"]]
        .style.format("{:.4f}")
        .set_caption("Table 10a - Out-of-time test performance: threshold-free metrics"))
display(metrics[["Threshold", "Precision", "Recall", "F1", "Accuracy", "Cost/loan"]]
        .style.format({"Threshold": "{:.3f}", "Precision": "{:.3f}", "Recall": "{:.3f}",
                       "F1": "{:.3f}", "Accuracy": "{:.3f}", "Cost/loan": "{:.4f}"})
        .set_caption("Table 10b - Out-of-time test performance: at the cost-based operating point"))

**Table 10 — Out-of-time test performance, with the baselines included.**

Read it in this order.

**First, the `lift vs B1` columns.** This is the question notebook 01 §1.5 posed: does the
model beat the rule the platform already runs? A positive lift, in the third decimal or
better, is the project's actual contribution. A negative or negligible lift is a valid and
important result — it says `sub_grade` already captures the available signal, and the
honest recommendation becomes "keep the existing rule". Report it either way.

**Second, B0 as a floor.** Its PR-AUC sits at the prevalence and its ROC-AUC at 0.500,
confirming the metrics behave as notebook 01 Table 2 predicted.

**Third, the model ranking.** The gradient-boosted models should edge out logistic
regression on ROC-AUC and PR-AUC — but by thousandths, not tenths. On tabular credit data
with engineered features the linear baseline is genuinely competitive, which is much of why
production scorecards remain linear.

**Finally, accuracy.** Note how high it is for every model and how little it means:
predicting "nobody defaults" scores around 80 %. Recall at the cost-based threshold is far
more informative, and is deliberately high at the expense of precision because the cost
matrix in notebook 01 says a missed default hurts four times more than a wrongly rejected
applicant.

If any model shows ROC-AUC materially above ~0.75 on the real data, stop and look for a
leaked column before believing it.

**Action →** Carry the best model forward to the error analysis in 9.6; carry the lift over
B1 into the conclusions as the headline claim.

### 9.5 Figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for name, score in scores_test.items():
    fpr, tpr, _ = roc_curve(y_test, score)
    axes[0].plot(fpr, tpr, lw=2, color=MODEL_COLORS[name],
                 label=f"{name} (AUC {results[name]['ROC-AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], ls="--", lw=1.5, color=C_GRAY, label="Random (0.500)")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves (out-of-time test)")
axes[0].legend(frameon=False, loc="lower right", fontsize=9)

for name, score in scores_test.items():
    prec, rec, _ = precision_recall_curve(y_test, score)
    axes[1].plot(rec, prec, lw=2, color=MODEL_COLORS[name],
                 label=f"{name} (AP {results[name]['PR-AUC']:.3f})")
axes[1].axhline(y_test.mean(), ls="--", lw=1.5, color=C_GRAY,
                label=f"Prevalence ({y_test.mean():.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-recall curves (out-of-time test)")
axes[1].legend(frameon=False, loc="upper right", fontsize=9)

plt.tight_layout()
save_fig("08_roc_and_pr_curves")

**Figure 8 — ROC curves (left) and precision–recall curves (right) on the
out-of-time test set.**

The two panels are the same information under different framings, and the difference
matters. The ROC curves sit comfortably above the diagonal and look reassuring. The PR
curves are the sterner test: their baseline is the prevalence line (~0.20), not 0.5, and
precision falls away quickly as recall increases.

That decay is the real operating constraint. Catching most defaulters means accepting that
a large share of the applicants flagged will have repaid — the flat region on the right of
the PR curve is where a high-recall lending policy would sit. This is why PR-AUC is the
more informative headline for an imbalanced problem, and why reporting ROC-AUC alone
overstates practical usefulness.

The curves for the different models are close together, consistent with Table 3: model
choice matters much less on this problem than feature quality and threshold policy.

In [ ]:
# Restrict to actual models -- a baseline could top the table, and B1 doing so would be
# a finding to report, not a model to deploy.
best_model = metrics.loc[metrics.index.isin(calibrated_models)]["PR-AUC"].idxmax()
print(f"Best model by PR-AUC: {best_model}")
if metrics["PR-AUC"].idxmax() not in calibrated_models:
    print(f"!! NOTE: {metrics['PR-AUC'].idxmax()} outscores every fitted model on PR-AUC.")
    print("!! Report this honestly -- the incumbent rule is not beaten.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# --- Calibration: raw vs calibrated for the best model ------------------------------
for label, score, colour, style in [
        ("Uncalibrated", scores_test_raw[best_model], C_ORANGE, "--"),
        ("Calibrated (isotonic)", scores_test[best_model], C_BLUE, "-")]:
    frac_pos, mean_pred = calibration_curve(y_test, score, n_bins=12, strategy="quantile")
    brier = brier_score_loss(y_test, score)
    axes[0].plot(mean_pred, frac_pos, marker="o", ms=6, lw=2, ls=style, color=colour,
                 label=f"{label} (Brier {brier:.4f})")
axes[0].plot([0, 1], [0, 1], ls=":", lw=1.5, color=C_GRAY, label="Perfect calibration")
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed default rate")
axes[0].set_title(f"Calibration -- {best_model}")
axes[0].legend(frameon=False, fontsize=9)

# --- Score distribution by true outcome --------------------------------------------
for outcome, colour, label in [(0, C_BLUE, "Fully Paid"), (1, C_RED, "Charged Off")]:
    axes[1].hist(scores_test[best_model][y_test.to_numpy() == outcome], bins=40,
                 density=True, alpha=0.55, color=colour, label=label)
axes[1].axvline(OPERATING_THRESHOLD[best_model], color="#0b0b0b", lw=1.8, ls="--",
                label=f"Operating threshold ({OPERATING_THRESHOLD[best_model]:.3f})")
axes[1].set_xlabel("Predicted probability of default")
axes[1].set_ylabel("Density")
axes[1].set_title(f"Score distribution by outcome -- {best_model}")
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
save_fig("09_calibration_and_scores")

**Figure 9 — Calibration curve before and after isotonic calibration (left) and the
calibrated score distribution split by true outcome (right), best model.**

The left panel shows why calibration earned its place in the pipeline. The uncalibrated
curve sits well below the diagonal across most of the range — the observed default
rate is far lower than the predicted probability — the class-weighted model
systematically **overstates** default probability, exactly as the reweighting of the class
prior predicts. Isotonic regression pulls it back toward the diagonal and the Brier score
improves. ROC-AUC is unchanged by this, because calibration is monotone and cannot alter
the ranking; that is precisely the point, and precisely why AUC alone is insufficient for a
model whose probabilities get used.

The right panel shows the separation the metrics summarise. The two distributions are
clearly shifted apart but overlap heavily — there is no threshold that cleanly divides
them. The dashed line marks the cost-based operating point; everything to the right is
declined. Reading the overlap around that line gives an intuitive sense of the errors the
policy accepts, in a way a single F1 number does not.

In [ ]:
cm = confusion_matrix(y_test, (scores_test[best_model] >= OPERATING_THRESHOLD[best_model]).astype(int))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

from matplotlib.colors import LinearSegmentedColormap
sequential_blue = LinearSegmentedColormap.from_list(
    "blue_seq", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281"])

fig, ax = plt.subplots(figsize=(5.6, 4.6))
labels = np.array([[f"{cm[i, j]:,}\n({cm_norm[i, j]:.1%})" for j in range(2)] for i in range(2)])
sns.heatmap(cm_norm, annot=labels, fmt="", cmap=sequential_blue, vmin=0, vmax=1,
            cbar_kws={"label": "Share of true class"}, linewidths=2, linecolor="#fcfcfb",
            xticklabels=["Predicted:\nFully Paid", "Predicted:\nCharged Off"],
            yticklabels=["Actual:\nFully Paid", "Actual:\nCharged Off"], ax=ax)
ax.set_title(f"Confusion matrix -- {best_model}\n(row-normalised, cost-based threshold)")
save_fig("10_confusion_matrix")

**Figure 10 — Row-normalised confusion matrix for the best model at the cost-based
threshold.**

Rows are normalised so each row shows what happens to that true class: the bottom-right
cell is recall (defaulters correctly flagged) and the top-right is the false-positive rate
(good applicants wrongly declined).

The asymmetry is deliberate, not a defect. The cost matrix says a missed default costs four
times a wrongly rejected applicant, so the optimiser buys a high catch rate by accepting
many false positives. Whether that trade is acceptable is a business question — with the
placeholder costs replaced by real ones, this single figure is what a credit committee
would actually argue over.

Note also that the bottom-left cell — defaults the model let through — is where the
realised losses live. Reducing it is worth far more than a marginal AUC improvement.

### 9.6 Error analysis — where does the model fail?

> *Which segments does the model get wrong? Is the error systematic or random?*

An aggregate ROC-AUC hides everything that matters operationally. A model can post a
respectable overall number while being badly wrong for one grade band, one loan purpose,
or — most consequentially — for the most recent months of the test period.

The helper below slices the test set and recomputes performance within each segment. Two
readings matter:

- **Systematic error** — a segment where the model is consistently worse. That is a
  modelling gap: a missing feature, or a population the training data underrepresents.
- **Random error** — a segment that looks worse purely because it is small. The `n` column
  is printed for exactly this reason; a segment of 200 loans will swing wildly.

The **issue-quarter** slice is the one that feeds Step 11 directly: if performance decays
across the test period, that is the retraining signal, measured rather than assumed.

In [ ]:
def segment_report(segment: pd.Series, y_true, y_score, threshold: float,
                   min_n: int = 100) -> pd.DataFrame:
    """Per-segment performance. Segments smaller than min_n are dropped as unreliable."""
    rows = []
    y_pred = (y_score >= threshold).astype(int)
    for value, idx in pd.Series(range(len(segment)), index=segment.values).groupby(level=0):
        pos = idx.to_numpy()
        yt, ys, yp = y_true[pos], y_score[pos], y_pred[pos]
        if len(pos) < min_n or len(np.unique(yt)) < 2:
            continue
        tn = int(((yp == 0) & (yt == 0)).sum()); fp = int(((yp == 1) & (yt == 0)).sum())
        fn = int(((yp == 0) & (yt == 1)).sum()); tp = int(((yp == 1) & (yt == 1)).sum())
        rows.append({
            "segment": value, "n": len(pos), "default_rate": yt.mean(),
            "ROC-AUC": roc_auc_score(yt, ys),
            "FPR": fp / max(fp + tn, 1),      # good applicants wrongly declined
            "FNR": fn / max(fn + tp, 1),      # defaults the model let through
            "decline_rate": yp.mean(),
        })
    return pd.DataFrame(rows).set_index("segment")


y_test_np = y_test.to_numpy()
best_scores = scores_test[best_model]
best_thr = OPERATING_THRESHOLD[best_model]

test_meta = df.loc[~train_mask].reset_index(drop=True)
segments = {
    "grade": test_meta["grade"],
    "purpose": test_meta["purpose"],
    "term": test_meta["term"].astype(int).astype(str) + " months",
    "FICO band": pd.cut(test_meta["fico_avg"], [0, 680, 700, 720, 750, 900],
                        labels=["<680", "680-700", "700-720", "720-750", "750+"]),
    "issue quarter": test_meta["issue_d"].dt.to_period("Q").astype(str),
}

segment_tables = {}
for label, seg in segments.items():
    table = segment_report(seg, y_test_np, best_scores, best_thr)
    if len(table):
        segment_tables[label] = table
        display(table.style.format({
            "n": "{:,.0f}", "default_rate": "{:.3f}", "ROC-AUC": "{:.3f}",
            "FPR": "{:.3f}", "FNR": "{:.3f}", "decline_rate": "{:.3f}"})
            .set_caption(f"Table 11 - Error analysis by {label} ({best_model})"))

In [ ]:
# The time slice deserves its own figure: it is the retraining signal for Step 11.
if "issue quarter" in segment_tables:
    q = segment_tables["issue quarter"]
    fig, axes = plt.subplots(2, 1, figsize=(9.5, 6), sharex=True)

    axes[0].plot(q.index, q["ROC-AUC"], color=C_BLUE, lw=2, marker="o", ms=6)
    axes[0].axhline(results[best_model]["ROC-AUC"], color=C_GRAY, ls="--", lw=1.5,
                    label=f"overall {results[best_model]['ROC-AUC']:.3f}")
    axes[0].set_ylabel("ROC-AUC")
    axes[0].set_title(f"Performance across the test period -- {best_model}")
    axes[0].legend(frameon=False, fontsize=9)

    axes[1].plot(q.index, q["FNR"], color=C_RED, lw=2, marker="o", ms=6,
                 label="FNR (defaults missed)")
    axes[1].plot(q.index, q["FPR"], color=C_ORANGE, lw=2, marker="s", ms=6,
                 label="FPR (good applicants declined)")
    axes[1].set_ylabel("Error rate")
    axes[1].set_xlabel("Issue quarter")
    axes[1].legend(frameon=False, fontsize=9)
    plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")
    plt.tight_layout()
    save_fig("11_error_by_quarter")

**Figure 11 — Model performance and error rates across the test period.**

This is the single most operationally useful figure in the notebook, because it answers a
question the aggregate metrics cannot: *is the model still working at the end of the test
window, or only at the start?*

A **downward slope in ROC-AUC** is concept drift made visible — the relationship the model
learned in the training period is decaying as it moves further from it. A **rising FNR**
is the expensive version of the same thing: more defaults slipping through, quarter by
quarter.

Read the right-hand edge with care, though. Notebook 02 Figure 7 showed the
right-censoring artefact — the final quarters contain a non-random subsample of loans that
resolved early. Some of the movement there is that artefact, not genuine decay.

**Action →** Use the observed decay rate to set the retraining cadence in notebook 04. If
ROC-AUC falls by more than a few points across the test window, a fixed annual refit is
too slow.

### 9.7 Fairness — who bears the errors?

> *Check fairness across groups if the decision affects people.*

Lending decisions affect people directly, and in several jurisdictions fairness testing is
a legal precondition rather than good practice. This section measures it.

**An important caveat, stated up front.** This dataset contains **no protected
attributes** — no race, gender, age or ethnicity. What follows uses **proxies**: US region
(a coarse and imperfect correlate of demographics through residential segregation), income
band, and housing status. So this is an **indicative** screen, not a compliance audit. It
can reveal a disparity worth investigating; it cannot certify that none exists.

Three quantities per group:

| Measure | Meaning | Why it matters |
|---|---|---|
| **Decline rate** | Share of applicants the policy rejects | The decision people actually experience |
| **FPR** | Good applicants wrongly declined | Creditworthy people denied credit — the harm of a false accusation |
| **FNR** | Defaults wrongly approved | Borrowers set up to fail, plus lender loss |

The **disparate-impact ratio** (lowest decline rate ÷ highest, conventionally compared
against 0.8) summarises the spread. Note it is computed on a *decline* rate, so we compare
the most-favoured group against the least-favoured.

In [ ]:
US_REGIONS = {
    "CA": "West", "OR": "West", "WA": "West", "NV": "West", "AZ": "West", "CO": "West",
    "NY": "Northeast", "NJ": "Northeast", "PA": "Northeast", "MA": "Northeast",
    "CT": "Northeast", "TX": "South", "FL": "South", "GA": "South", "NC": "South",
    "VA": "South", "TN": "South", "IL": "Midwest", "OH": "Midwest", "MI": "Midwest",
    "WI": "Midwest", "MN": "Midwest", "IN": "Midwest", "MO": "Midwest",
}

fairness_groups = {
    "US region (proxy)": test_meta["addr_state"].map(US_REGIONS).fillna("Other"),
    "Income band": pd.qcut(test_meta["annual_inc"], 4,
                           labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"]),
    "Housing status": test_meta["home_ownership"],
}

fairness_tables = {}
for label, groups in fairness_groups.items():
    table = segment_report(groups, y_test_np, best_scores, best_thr, min_n=200)
    if len(table) < 2:
        continue
    ratio = table["decline_rate"].min() / table["decline_rate"].max()
    fairness_tables[label] = (table, ratio)
    flag = "PASS (>= 0.8)" if ratio >= 0.8 else "FLAG (< 0.8) -- investigate"
    display(table[["n", "default_rate", "decline_rate", "FPR", "FNR", "ROC-AUC"]]
            .style.format({"n": "{:,.0f}", "default_rate": "{:.3f}",
                           "decline_rate": "{:.3f}", "FPR": "{:.3f}",
                           "FNR": "{:.3f}", "ROC-AUC": "{:.3f}"})
            .set_caption(f"Table 12 - Fairness screen by {label}: "
                         f"disparate-impact ratio {ratio:.3f} -- {flag}"))

In [ ]:
if fairness_tables:
    fig, axes = plt.subplots(1, len(fairness_tables),
                             figsize=(5.2 * len(fairness_tables), 4.3), squeeze=False)
    for ax, (label, (table, ratio)) in zip(axes[0], fairness_tables.items()):
        x = np.arange(len(table)); w, gap = 0.26, 0.012
        ax.bar(x - w - gap, table["decline_rate"], w, color=C_BLUE, label="Decline rate")
        ax.bar(x, table["FPR"], w, color=C_ORANGE, label="FPR")
        ax.bar(x + w + gap, table["FNR"], w, color=C_AQUA, label="FNR")
        ax.set_xticks(x)
        ax.set_xticklabels(table.index, rotation=20, ha="right", fontsize=9)
        ax.set_title(f"{label}\ndisparate impact {ratio:.3f}", fontsize=11)
        ax.set_ylabel("Rate")
        ax.legend(frameon=False, fontsize=8.5)
    plt.tight_layout()
    save_fig("12_fairness_screen")

**Figure 12 / Table 12 — Fairness screen across proxy groups.**

The decline-rate bars show what different groups actually experience from the policy; the
FPR and FNR bars decompose *why*.

The interpretation turns on one distinction. A group with a higher decline rate **and** a
correspondingly higher realised default rate is being treated consistently — the model is
tracking genuine risk. A group with a higher decline rate **but a similar default rate and
a higher FPR** is being penalised by the model rather than by its own risk: creditworthy
applicants declined at an elevated rate. The second pattern is the one that warrants
action, and looking at decline rates alone cannot tell the two apart.

The income-band slice will almost always show a large spread, and that is expected rather
than alarming: income is a legitimate, directly causal underwriting factor, and
`loan_to_income` is deliberately one of our features. The region slice is the more
diagnostic one, since geography has no causal claim on repayment beyond what the other
features already capture.

**Action →** Any group where the ratio falls below 0.8 **with** an elevated FPR goes on the
list for investigation before deployment: consider dropping `addr_state`, or testing a
group-blind variant and comparing both the fairness spread and the loss of performance.

### 9.8 How optimistic would a random split have been?

Everything above used the out-of-time protocol. This section re-runs the pipeline **once**
under a random stratified split — the design most public Kaggle notebooks on this dataset
use — purely to quantify the difference.

This is a headline result of the project, not an aside. If random-split scores are
materially higher, then published leaderboard numbers on this dataset are not comparable to
an honest forecasting estimate, and the gap is the size of the illusion.

The out-of-time number remains the one we report and defend.

In [ ]:
ref_train_mask = df["is_train_random_ref"].to_numpy()
Xr_train, Xr_test = X.loc[ref_train_mask], X.loc[~ref_train_mask]
yr_train, yr_test = y.loc[ref_train_mask], y.loc[~ref_train_mask]

comparison = []
for name, model in models.items():
    ref_fit = model.fit(Xr_train, yr_train)            # same pipeline, random split
    ref_score = ref_fit.predict_proba(Xr_test)[:, 1]
    comparison.append({
        "model": name,
        "OOT ROC-AUC": results[name]["ROC-AUC"],
        "Random ROC-AUC": roc_auc_score(yr_test, ref_score),
        "OOT PR-AUC": results[name]["PR-AUC"],
        "Random PR-AUC": average_precision_score(yr_test, ref_score),
    })

split_comparison = pd.DataFrame(comparison).set_index("model")
split_comparison["ROC-AUC optimism"] = (
    split_comparison["Random ROC-AUC"] - split_comparison["OOT ROC-AUC"])
split_comparison["PR-AUC optimism"] = (
    split_comparison["Random PR-AUC"] - split_comparison["OOT PR-AUC"])

display(split_comparison.style.format("{:.4f}").set_caption(
    "Out-of-time vs random stratified split -- the cost of look-ahead"))

# Refit the reported models on the proper training data so nothing downstream is polluted.
for name, model in models.items():
    model.fit(X_fit, y_fit)
print("\nModels refitted on the out-of-time training slice.")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.4))
x = np.arange(len(split_comparison))
w, gap = 0.38, 0.014          # gap keeps a surface sliver between adjacent bars
ax.bar(x - w / 2 - gap, split_comparison["OOT ROC-AUC"], w, color=C_BLUE,
       label="Out-of-time (reported)")
ax.bar(x + w / 2 + gap, split_comparison["Random ROC-AUC"], w, color=C_ORANGE,
       label="Random split (optimistic)")
for i, (oot, rnd) in enumerate(split_comparison[["OOT ROC-AUC", "Random ROC-AUC"]].values):
    ax.annotate(f"{oot:.3f}", (i - w / 2 - gap, oot), ha="center", va="bottom", fontsize=9,
                color="#52514e")
    ax.annotate(f"{rnd:.3f}", (i + w / 2 + gap, rnd), ha="center", va="bottom", fontsize=9,
                color="#52514e")
ax.axhline(0.5, color=C_GRAY, ls="--", lw=1.5, label="Random guessing")
ax.set_xticks(x)
ax.set_xticklabels(split_comparison.index, rotation=12, ha="right")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.45, max(split_comparison[["OOT ROC-AUC", "Random ROC-AUC"]].max()) * 1.09)
ax.set_title("ROC-AUC under both evaluation protocols, by model")
ax.legend(frameon=False, fontsize=9)
save_fig("13_split_protocol_comparison")

**Figure 13 — The same models scored under both evaluation protocols.**

The spread between the blue and orange bars for a single model should be compared with the
spread *across* models within either colour. Where the protocol gap is the larger of the
two — which is the usual finding on this dataset — the conclusion is blunt: **how you
split matters more than which model you fit.**

That reframes what a leaderboard-style comparison is worth. A notebook reporting a higher
AUC under a random split has not necessarily built a better model; it has measured a
different, easier problem. It is also the strongest argument for the shared-split
requirement in the brief — without a fixed protocol, group-to-group numbers are not
comparable.

If the gap in your run is negligible, say so plainly: it would mean drift over this window
is mild, which is itself a finding worth reporting rather than hiding.

## Step 10 — Explain the model

### 10.1 Permutation importance and coefficients

Two complementary views before SHAP: permutation importance says *how much the model
relies* on a column, the logistic coefficients say *in which direction and how strongly*
a linear model uses it.

In [ ]:
# Permutation importance: model-agnostic, and measured on the TEST set because we are
# asking "what did this model rely on?", not making a modelling choice from it.
perm_n = min(20_000, len(X_test))
perm_idx = np.random.default_rng(RANDOM_SEED).choice(len(X_test), perm_n, replace=False)

perm = permutation_importance(
    calibrated_models[best_model], X_test.iloc[perm_idx], y_test.iloc[perm_idx],
    scoring="roc_auc", n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1)

importance = (pd.Series(perm.importances_mean, index=X_test.columns)
                .sort_values(ascending=False).head(15))
errors = pd.Series(perm.importances_std, index=X_test.columns).loc[importance.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

axes[0].barh(importance.index[::-1], importance.values[::-1],
             xerr=errors.values[::-1], color=C_BLUE, height=0.65,
             error_kw={"ecolor": C_GRAY, "lw": 1.2})
axes[0].set_xlabel("Drop in ROC-AUC when the column is shuffled")
axes[0].set_title(f"Permutation importance -- {best_model}")

# Logistic-regression coefficients, for interpretability alongside the black box.
lr = fitted["Logistic Regression"]
feature_names = lr.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=feature_names)
top_coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index).head(15)[::-1]
axes[1].barh(top_coefs.index, top_coefs.values,
             color=[C_RED if v > 0 else C_BLUE for v in top_coefs.values], height=0.65)
axes[1].axvline(0, color="#0b0b0b", lw=1)
axes[1].set_xlabel("Coefficient (standardised features; + raises default risk)")
axes[1].set_title("Logistic regression coefficients")

plt.tight_layout()
save_fig("14_feature_importance")

**Figure 14 — Permutation importance for the best model (left) and standardised
logistic-regression coefficients (right).**

The two panels are complementary: permutation importance says *how much the model relies*
on a column, the coefficients say *in which direction and how strongly* a linear model
uses it. The error bars on the left are the spread across five shuffles — a bar shorter
than its own error bar is not distinguishable from noise.

Expect `int_rate`, `sub_grade_ordinal` and `grade_ordinal` near the top. That deserves
scrutiny rather than satisfaction: those columns are **Lending Club's own risk
assessment**, not raw applicant data. To a large extent a high-importance score there
means our model has learned to reproduce their underwriting model. It is legitimate — the
grade genuinely is available at application time — but it caps how much credit we can take
for the predictive performance, and it is worth reporting the model refitted without the
grade/rate columns to see how much signal survives on applicant data alone.

Among the genuinely independent features, `loan_to_income`, `dti` and `fico_avg` carrying
weight is the expected and reassuring result: leverage, existing debt burden and credit
history are what underwriting theory says should matter.

The colour split on the right is directional: red raises modelled default risk, blue
lowers it. Signs should be checkable against intuition — higher FICO lowering risk, higher
leverage raising it. A sign that contradicts domain knowledge usually indicates
collinearity (recall `loan_amnt` and `installment` from Figure 5) rather than a discovery.

### 10.2 SHAP — per-prediction attribution

Permutation importance gives one number per feature for the whole model. **SHAP** gives a
contribution per feature **per applicant**, which is what makes it operationally different:
it can answer *"why was this particular application declined?"* — the question an
adverse-action notice legally has to answer in several jurisdictions.

SHAP values are additive: for each applicant they sum to the gap between that prediction
and the model's average prediction. The import is guarded, so the notebook still runs end
to end without `shap` installed.

Two practical notes. SHAP is computed on the **uncalibrated** tree model — calibration is a
monotone post-processing step that rescales scores without changing which features drove
them — and on a subsample, because exact values are expensive.

In [ ]:
SHAP_AVAILABLE = False
try:
    import shap
    SHAP_AVAILABLE = True
    print(f"shap {shap.__version__} available")
except ImportError:
    print("shap not installed -- skipping 10.2. Run `pip install shap` to enable it.")

In [ ]:
if SHAP_AVAILABLE:
    tree_models = ["XGBoost", "HistGradientBoosting", "Random Forest"]
    shap_target = next((m for m in tree_models if m in fitted), None)

    if shap_target is None:
        print("No tree model fitted; TreeExplainer not applicable.")
    else:
        pipe = fitted[shap_target]
        n_shap = min(2_000, len(X_test))
        idx = np.random.default_rng(RANDOM_SEED).choice(len(X_test), n_shap, replace=False)
        X_shap_raw = X_test.iloc[idx]

        # Push the raw frame through the FITTED preprocessor (transform only, never fit).
        X_shap = pipe.named_steps["prep"].transform(X_shap_raw)
        shap_features = list(pipe.named_steps["prep"].get_feature_names_out())

        explainer = shap.TreeExplainer(pipe.named_steps["clf"])
        shap_values = explainer.shap_values(X_shap)
        if isinstance(shap_values, list):          # some versions return one array per class
            shap_values = shap_values[1]
        if shap_values.ndim == 3:                  # (n, features, classes)
            shap_values = shap_values[:, :, 1]

        print(f"SHAP values: {shap_values.shape} from {shap_target}\n")
        shap.summary_plot(shap_values, X_shap, feature_names=shap_features,
                          max_display=15, show=False, plot_size=(9, 6))
        plt.title(f"SHAP summary -- {shap_target}", fontsize=12, fontweight="bold")
        save_fig("15_shap_summary")

**Figure 15 — SHAP summary (beeswarm) for the best tree model.**

Each dot is one applicant's contribution from one feature. Position on the x-axis is the
push on the prediction — right means "more likely to default" — and colour is the feature's
own value, red high and blue low.

The beeswarm shows three things the bar chart in Figure 14 cannot. **Direction**: a band
running from blue on the left to red on the right means high values raise risk, and the
sign should match domain intuition — high `loan_to_income` pushing right, high `fico_avg`
pushing left. **Magnitude spread**: a feature with a long tail matters enormously for a few
applicants and barely at all for the rest, which an average importance hides completely.
**Interactions**: vertical colour mixing at a given x-position means the feature's effect
depends on something else.

**Action →** Check every top feature's direction against underwriting knowledge. A
feature pushing the *wrong* way is a red flag for leakage or a data defect — per the
process, suspect the pipeline before believing you have found something new.

### 10.3 Partial dependence — the shape of each effect

SHAP explains individual predictions; **partial dependence** shows the average modelled
relationship across the whole range of a feature. It answers a different question: not
*"which features matter?"* but *"what does the model think the relationship looks like?"*

That shape is what makes a model arguable in front of a credit committee. A monotonically
rising default probability in `loan_to_income` is defensible. A jagged, non-monotonic curve
usually means overfitting or a data artefact, and would be a strong argument for imposing a
monotonic constraint.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

pdp_features = [f for f in ["loan_to_income", "dti", "fico_avg", "int_rate"]
                if f in X_test.columns][:3]
pdp_sample = X_test.sample(min(3_000, len(X_test)), random_state=RANDOM_SEED)

fig, axes = plt.subplots(1, len(pdp_features), figsize=(4.6 * len(pdp_features), 4))
PartialDependenceDisplay.from_estimator(
    calibrated_models[best_model], pdp_sample, pdp_features,
    ax=axes, line_kw={"color": C_BLUE, "lw": 2.2})
for ax, feature in zip(np.atleast_1d(axes), pdp_features):
    ax.set_title(feature, fontsize=11, fontweight="bold")
    ax.set_ylabel("Partial dependence\n(P(default))")
plt.suptitle(f"Partial dependence -- {best_model}", fontsize=12, fontweight="bold")
plt.tight_layout()
save_fig("16_partial_dependence")

In [ ]:
# One two-way plot: does the effect of leverage depend on credit quality?
if len(pdp_features) >= 2:
    pair = [(pdp_features[0], pdp_features[-1])]
    fig, ax = plt.subplots(figsize=(6.2, 5))
    PartialDependenceDisplay.from_estimator(
        calibrated_models[best_model], pdp_sample, pair, ax=ax)
    ax.set_title(f"Two-way partial dependence: {pair[0][0]} x {pair[0][1]}",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    save_fig("17_partial_dependence_2way")

**Figures 16 and 17 — Partial dependence, one-way and two-way.**

Read the one-way curves for **shape and monotonicity**. Default probability should rise
with `loan_to_income` and `dti` and fall with `fico_avg`. Where a curve flattens, the
feature has stopped adding information — often the point beyond which the population
thins out, so treat the extreme ends as unreliable rather than as findings.

The two-way plot asks whether the effect of one feature *depends* on another: if the
leverage gradient is steeper for low-FICO applicants than for high-FICO ones, the model has
learned an interaction, and that is precisely the kind of structure a linear scorecard
cannot represent without being told to. It is also the concrete justification for
preferring a tree ensemble — or for adding an explicit interaction term to the logistic
model and keeping the simpler, more auditable option.

**One caveat the process insists on:** partial dependence is a statement about **the
model**, not about the world. It marginalises over the other features and can extrapolate
into regions where no applicant actually exists. It shows what the model believes; it does
not establish causation.

**Action →** If any curve is non-monotonic where theory says it should not be, test a
monotonic-constraint variant (`monotone_constraints` in XGBoost / HistGB) — usually near-free
in performance, and much easier to defend.

## Conclusions

Fill in the specific numbers from your run; the structure below is what the report needs.

**Did we beat the incumbent?** This is the headline, and Table 10's `lift vs B1` column
answers it. `sub_grade` is Lending Club's own underwriting and it is a strong baseline; the
project's contribution is the margin above it, not the absolute AUC. If that margin is
negligible, the correct recommendation is to keep the existing rule — notebook 01 §1.5
asked whether this needs machine learning, and this is the number that answers it.

**Performance.** The fitted models land in a narrow band, with the gradient-boosted models
marginally ahead on PR-AUC and logistic regression close behind. On the real data expect
ROC-AUC around 0.68–0.72. That is the honest ceiling for application-time features on this
dataset, consistent with the published credit-scoring literature — default is driven
substantially by post-origination events (job loss, illness, divorce) that no application
form observes.

**Where the model fails.** Section 9.6 is more actionable than any aggregate metric. Check
the issue-quarter slice in particular: decay across the test period sets the retraining
cadence in notebook 04, and is the difference between a model that works and a model that
worked.

**Model choice is not the interesting variable.** The gap between the best and worst model
is small next to the gap between evaluation protocols (Figure 13) and next to the effect of
the threshold policy (Table 2). Effort spent on feature quality, split integrity and
cut-off economics pays far more than effort spent on hyperparameter search.

**Recommendation.** Logistic regression remains a defensible production choice: within
thousandths of the ensembles, auditable coefficient by coefficient, trivial to monitor, and
straightforward to justify to a regulator under adverse-action requirements. Deploy the
gradient-boosted model only if the measured lift survives on real data and the
interpretability cost is explicitly accepted.

**Calibration matters as much as ranking.** The uncalibrated models systematically
overstate default probability (Figure 9). Since expected loss is computed from the
probability itself, an uncalibrated model misprices every loan even when it ranks them
perfectly.

---

## Limitations

These constrain what may be claimed from the results above. They are not boilerplate.

**1. `issue_d` is a proxy for the application date.** It records when the loan was
*issued*, not when it was applied for or decided. Using it to derive
`credit_history_years` and to define the out-of-time cutoff is legitimate and is standard
practice on this dataset — but the true decision moment is not in the data. Any
application-to-issuance lag is invisible to us, so features are dated slightly later than
a live scorecard would observe them, and the cutoff separates *issuance* cohorts rather
than *decision* cohorts.

**2. Right-censoring from the terminal-status filter.** Keeping only `Fully Paid` and
`Charged Off` discards loans still `Current`. Because 36- and 60-month loans issued near
the end of the window have not had time to mature, the post-cutoff test set is enriched
with loans that resolved *early* — and early resolution is disproportionately charge-off.
The rising tail in Figure 7 is this artefact. Consequences: the test-period default rate is
**not** the true default rate for that cohort, and the out-of-time metrics are computed on
a non-random subsample of post-cutoff originations. A survival-analysis framing
(time-to-default with censoring handled explicitly) would address this properly and is the
natural extension of this work.

**3. Accepted loans only — selection bias.** The dataset contains only applicants Lending
Club approved. The model therefore estimates default risk **conditional on acceptance**,
not for the through-the-door population. Applying it to all applicants would extrapolate
beyond the region the training data covers. This is the classic reject-inference problem;
Lending Club publishes a rejected-applications file, but it lacks outcomes by
construction, so the problem cannot be solved simply by joining the two.

**4. Drift is measured, not corrected.** The out-of-time design quantifies how much
performance decays across time (Figure 13) but does nothing to counter it. A deployed
scorecard would need scheduled refitting, population-stability monitoring, and alerting on
score-distribution shift.

**5. Fairness is screened by proxy only.** Section 9.7 measures decline rate, FPR and FNR
across US region, income band and housing status, with a disparate-impact ratio. That is a
genuine screen, and it is no longer "not assessed" — but it is **not a compliance audit**.
The dataset contains no protected attributes (race, gender, age), so every grouping used is
a proxy, and region in particular is a weak and uneven correlate of demographics. A
disparity found here is real and worth investigating; an absence of disparity here does
**not** demonstrate the model is fair on the attributes that actually matter legally.
Deployment would require testing on real protected attributes, which this data cannot
support.

**6. Costs are placeholders.** `COST_FN = 1.0` and `COST_FP = 0.25` are illustrative. Every
point metric — precision, recall, F1, the operating threshold, `Cost/loan` — moves with
them. Replace them with real portfolio economics before quoting any of those numbers.

In [ ]:
print("=" * 72)
print("RUN SUMMARY".center(72))
print("=" * 72)
print(f"Data source      : {DATA_SOURCE}")
if DATA_SOURCE == "synthetic":
    print("                   ^ SMOKE TEST ONLY -- not findings. See README.md.")
print(f"Split            : out-of-time, cutoff {CUTOFF_DATE.date()}")
print(f"Train / test     : {len(X_train):,} / {len(X_test):,}")
print(f"Best model       : {best_model}")
print(f"  ROC-AUC (OOT)  : {results[best_model]['ROC-AUC']:.4f}")
print(f"  PR-AUC  (OOT)  : {results[best_model]['PR-AUC']:.4f}")
print(f"  KS      (OOT)  : {results[best_model]['KS']:.4f}")
print(f"  Brier   (OOT)  : {results[best_model]['Brier']:.4f}")
b1_key = "B1 - sub_grade (incumbent)"
if b1_key in results:
    print(f"  vs incumbent B1: ROC-AUC {results[best_model]['ROC-AUC'] - results[b1_key]['ROC-AUC']:+.4f}, "
          f"PR-AUC {results[best_model]['PR-AUC'] - results[b1_key]['PR-AUC']:+.4f}")
print(f"\nNext: run notebooks/04_packaging_and_monitoring.ipynb")
print(f"\nArtifacts written to {ARTIFACTS}:")
for path in sorted(ARTIFACTS.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(ARTIFACTS)}")